# COVER-KBC Profile F1 Full TEST Run

This notebook runs the full blind TEST split once with Profile F1.

It does not evaluate TEST locally. It writes a 475-row `predictions.jsonl` and packages a leaderboard zip.

Important: Colab can only pull code that has been committed and pushed to GitHub. If the Profile F1 changes are still only local, commit/push them before running this notebook.

In [ ]:
# CELL 0 - Runtime and paths
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/vquclinh/FactElicit-AKBC"
REPO = Path("/content/FactElicit-AKBC")
CONFIG_PATH = Path("configs/experiments/cover_kbc_v3_8_profile_f1_stock_empty_rescue_test.yaml")
OUT_DIR = Path("/content/profile_f1_full_test_run")
SUBMISSION_ZIP = Path("/content/profile_f1_full_test_submission.zip")
DRIVE_OUT = Path("/content/drive/MyDrive/cover_kbc/profile_f1_full_test")

subprocess.run(["nvidia-smi"], check=False)
print("Python:", sys.version)
print("Output directory:", OUT_DIR)
print("Submission zip:", SUBMISSION_ZIP)
print("Drive output directory:", DRIVE_OUT)

In [ ]:
# CELL 0B - Mount Google Drive
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_OUT = Path("/content/drive/MyDrive/cover_kbc/profile_f1_full_test")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

print("Drive output directory:", DRIVE_OUT)

In [ ]:
# CELL 1 - Clone or pull the latest repo code
if REPO.exists():
    os.chdir(REPO)
    subprocess.check_call(["git", "status", "--short"])
    subprocess.check_call(["git", "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])

os.chdir(REPO)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Working directory:", Path.cwd())
print("Git HEAD:", head)

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Missing {CONFIG_PATH}. Commit/push the Profile F1 code first, then rerun this cell."
    )

In [ ]:
# CELL 2 - Install package and model dependencies
os.chdir(REPO)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[hf]"])
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "bitsandbytes",
    "accelerate",
    "huggingface_hub",
    "mistral-common>=1.6.2",
])

SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import cover_kbc
print("cover_kbc:", cover_kbc.__file__)
print("sys.path[0]:", sys.path[0])

In [ ]:
# CELL 3 - Hugging Face login
import getpass

token = os.environ.get("HF_TOKEN")

if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception as exc:
        print("Colab secret read failed:", repr(exc))

if not token:
    token = getpass.getpass("Paste HF_TOKEN: ")

os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token

from huggingface_hub import login
login(token=token, add_to_git_credential=False)

print("HF token is available.")

In [ ]:
# CELL 4 - Cheap preflight checks before loading weights
import yaml

os.chdir(REPO)
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

assert cfg["experiment"]["name"] == "cover_kbc_v3_8_profile_f1_stock_empty_rescue_test"
assert cfg["experiment"]["split"] == "test"
assert cfg["experiment"]["frozen_baseline"]["hidden_test_overall_f1"] == 0.5878
assert cfg["leaderboard_repair"]["profile"] == "F1_STOCK_EMPTY_RESCUE"
assert cfg["leaderboard_repair"]["features"]["mistral_stock_empty_rescue"] is True
assert cfg["leaderboard_repair"]["max_calls_by_relation"]["companyTradesAtStockExchange"] == 4

model_profile_text = json.dumps(cfg["model_profile"], sort_keys=True)
assert "Qwen" not in model_profile_text
assert "mistralai/Mistral-Small-3.2-24B-Instruct-2506" in model_profile_text
assert cfg["budget_assertion"]["total_published_parameters"] == 24011361280

subprocess.check_call([sys.executable, "scripts/audit_model_budget.py", str(CONFIG_PATH)])

print("Profile F1 preflight passed.")

In [ ]:
# CELL 5 - Run the full blind TEST split once with live logs
os.chdir(REPO)

if OUT_DIR.exists():
    raise FileExistsError(
        f"{OUT_DIR} already exists. Move/delete it manually if you intentionally want a fresh full TEST run."
    )

LOG_PATH = Path("/content/profile_f1_full_test_run.log")

cmd = [
    sys.executable,
    "-u",
    "scripts/run_cover.py",
    "--config",
    str(CONFIG_PATH),
    "--split",
    "test",
    "--output-dir",
    str(OUT_DIR),
    "--no-eval",
]

env = dict(os.environ)
env["PYTHONUNBUFFERED"] = "1"

print("Running:", " ".join(cmd))
print("Log path:", LOG_PATH)

with LOG_PATH.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        cmd,
        cwd=str(REPO),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()

    returncode = process.wait()

print("Return code:", returncode)
if returncode != 0:
    raise SystemExit(returncode)

In [ ]:
# CELL 6 - Validate output shape and repair accounting
PREDICTIONS = OUT_DIR / "predictions.jsonl"
MANIFEST = OUT_DIR / "manifest.json"
REPAIR_ACCOUNTING = OUT_DIR / "repair_accounting.json"
RUN_ACCOUNTING = OUT_DIR / "run_accounting.json"
CALLS = OUT_DIR / "calls.jsonl"

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

assert OUT_DIR.exists(), f"Missing output directory: {OUT_DIR}"
assert PREDICTIONS.exists(), f"Missing predictions: {PREDICTIONS}"
assert MANIFEST.exists(), f"Missing manifest: {MANIFEST}"
assert REPAIR_ACCOUNTING.exists(), f"Missing repair accounting: {REPAIR_ACCOUNTING}"
assert RUN_ACCOUNTING.exists(), f"Missing run accounting: {RUN_ACCOUNTING}"
assert CALLS.exists(), f"Missing calls log: {CALLS}"
assert not (OUT_DIR / "metrics.json").exists(), "TEST is blind; metrics.json should not exist."

rows = read_jsonl(PREDICTIONS)
assert len(rows) == 475, len(rows)

relations = {}
empty_rows = 0
for row in rows:
    assert set(row) == {"SubjectEntity", "Relation", "ObjectEntities"}, row
    assert isinstance(row["SubjectEntity"], str)
    assert isinstance(row["Relation"], str)
    assert isinstance(row["ObjectEntities"], list)
    assert all(isinstance(item, str) for item in row["ObjectEntities"])
    relations[row["Relation"]] = relations.get(row["Relation"], 0) + 1
    if not row["ObjectEntities"]:
        empty_rows += 1

expected_relations = {
    "awardWonBy": 10,
    "companyTradesAtStockExchange": 100,
    "countryLandBordersCountry": 67,
    "hasArea": 100,
    "hasCapacity": 98,
    "personHasCityOfDeath": 100,
}
assert relations == expected_relations, relations

run_accounting = json.loads(RUN_ACCOUNTING.read_text(encoding="utf-8"))
repair = json.loads(REPAIR_ACCOUNTING.read_text(encoding="utf-8"))

assert run_accounting["total_queries"] == 475
assert run_accounting["successful_queries"] == 475
assert run_accounting["failed_queries"] == 0
assert repair["profile"] == "F1_STOCK_EMPTY_RESCUE"

stock = repair["by_relation"]["companyTradesAtStockExchange"]["stock_empty_rescue"]

print("Validation passed.")
print("Predictions:", PREDICTIONS)
print("Rows:", len(rows))
print("Relations:", relations)
print("Empty rows:", empty_rows)
print("Repair profile:", repair["profile"])
print("Total repair calls:", repair["total_repair_calls"])
print("Stock empty rescue:")
print(json.dumps(stock, indent=2, sort_keys=True))

In [ ]:
# CELL 7 - Package the TEST submission zip
os.chdir(REPO)

if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()

subprocess.check_call([
    sys.executable,
    "scripts/package_submission.py",
    "--predictions",
    str(PREDICTIONS),
    "--input",
    "benchmark/data/test.jsonl",
    "--split",
    "test",
    "--out",
    str(SUBMISSION_ZIP),
])

print("Submit this zip:", SUBMISSION_ZIP)
print("Raw predictions:", PREDICTIONS)

In [ ]:
# CELL 8 - Save all artifacts to Google Drive
from pathlib import Path

DRIVE_OUT = Path("/content/drive/MyDrive/cover_kbc/profile_f1_full_test")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_DIR = DRIVE_OUT / OUT_DIR.name

if DRIVE_RUN_DIR.exists():
    shutil.rmtree(DRIVE_RUN_DIR)

shutil.copytree(OUT_DIR, DRIVE_RUN_DIR)
shutil.copy2(SUBMISSION_ZIP, DRIVE_OUT / SUBMISSION_ZIP.name)

LOG_PATH = Path("/content/profile_f1_full_test_run.log")
if LOG_PATH.exists():
    shutil.copy2(LOG_PATH, DRIVE_OUT / LOG_PATH.name)

summary = {
    "repo_head": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "config": str(CONFIG_PATH),
    "output_dir": str(OUT_DIR),
    "drive_run_dir": str(DRIVE_RUN_DIR),
    "submission_zip": str(SUBMISSION_ZIP),
    "drive_submission_zip": str(DRIVE_OUT / SUBMISSION_ZIP.name),
    "predictions": str(PREDICTIONS),
    "manifest": str(MANIFEST),
    "repair_accounting": str(REPAIR_ACCOUNTING),
    "run_accounting": str(RUN_ACCOUNTING),
}
(DRIVE_OUT / "profile_f1_full_test_summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("Saved full run directory to:", DRIVE_RUN_DIR)
print("Saved submission zip to:", DRIVE_OUT / SUBMISSION_ZIP.name)
print("Saved summary to:", DRIVE_OUT / "profile_f1_full_test_summary.json")

In [ ]:
# CELL 9 - Download artifacts
from google.colab import files

files.download(str(SUBMISSION_ZIP))
files.download(str(PREDICTIONS))
files.download(str(MANIFEST))
files.download(str(REPAIR_ACCOUNTING))

In [ ]:
# CELL 10 - Disconnect runtime
from google.colab import runtime

runtime.unassign()